In [10]:
# 1. Install necessary libraries
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00


In [11]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)
from torchvision.transforms import (
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomResizedCrop,
    ToTensor,
)
import evaluate


In [ ]:
# 2. Define Model & Dataset Names
# We use the standard PlantVillage dataset from Hugging Face
dataset_name = "BrandonFors/Plant-Diseases-PlantVillage-Dataset"
# We use a standard pre-trained Vision Transformer
model_name = "google/vit-base-patch16-224-in21k"

# 3. Load Dataset
# This dataset is large, so we'll just use a small part for a quick demo.
# For your real project, remove the .select(range(1000)) parts!
train_ds = load_dataset(dataset_name, split="train").shuffle(seed=42).select(range(1000))
val_ds = load_dataset(dataset_name, split="test").shuffle(seed=42).select(range(500))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/321M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/362M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/170M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43456 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10849 [00:00<?, ? examples/s]

In [ ]:
# 4. Get Labels
labels = train_ds.features["label"].names
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}
num_labels = len(labels)
print(f"Total labels: {num_labels}. Example: {labels[:5]}")


Total labels: 38. Example: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']


In [ ]:
# 5. Load Image Processor (Handles normalization, resizing)
processor = AutoImageProcessor.from_pretrained(model_name)

# 6. Define Image Transformations
# We apply standard data augmentation
normalize = Normalize(mean=processor.image_mean, std=processor.image_std)
_transforms = Compose(
    [
        RandomResizedCrop(processor.size["height"]),
        RandomHorizontalFlip(),
        ToTensor(),
        normalize,
    ]
)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [ ]:
def preprocess_func(examples):
    """Apply transforms to a batch of images."""
    # The dataset has images under the 'image' key
    examples["pixel_values"] = [
        _transforms(image.convert("RGB")) for image in examples["image"]
    ]
    return examples


In [ ]:
# 7. Apply Transformations to the Datasets
# We use .set_transform() for memory efficiency
train_ds.set_transform(preprocess_func)
val_ds.set_transform(preprocess_func)


In [ ]:
(next(x.shape, y.shape) for x,y in train_ds.features.items())

<generator object <genexpr> at 0x79bb7045b920>

In [ ]:
# 8. Load the Pre-trained Model
model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    # This is crucial for replacing the head
    ignore_mismatched_sizes=True,
)


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 9. Define Training Configuration
# This is where you set hyperparameters
training_args = TrainingArguments(
    output_dir="./vit-plant-disease-results",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",  # Corrected argument name
    save_strategy="epoch",
    num_train_epochs=10, # Set to more (e.g., 10) for a real run
    learning_rate=2e-5,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,
    push_to_hub=False, # Set to True if you want to upload it
    report_to="none",
)

In [ ]:
# 10. Define Metrics Calculation
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Compute accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

# This function is needed to correctly format the data for the trainer
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.tensor([x["label"] for x in batch]),
    }


In [ ]:
# 11. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=processor, # The processor is passed as the tokenizer
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)


/tmp/ipython-input-1131766520.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 12. Start Training!
print("Starting training...")
trainer.train()
print("Training finished.")


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,3.347700,3.327818,0.314000
2,3.053400,3.057803,0.404000
3,2.802300,2.857009,0.456000
4,2.607000,2.695507,0.526000
5,2.504600,2.560303,0.570000
6,2.308200,2.488049,0.624000
7,2.262500,2.411095,0.644000
8,2.173500,2.361363,0.680000
9,2.193600,2.318933,0.682000
10,2.073100,2.308648,0.694000


Training finished.


In [ ]:
# 13. Evaluate the final model
print("Evaluating final model...")
eval_results = trainer.evaluate()
print(eval_results)


Evaluating final model...


{'eval_loss': 2.311532497406006, 'eval_accuracy': 0.7, 'eval_runtime': 6.6417, 'eval_samples_per_second': 75.282, 'eval_steps_per_second': 2.409, 'epoch': 10.0}


In [ ]:
# 14. (Optional) Run an inference
print("\n--- Running Inference Example ---")
# Let's grab one image from the validation set
one_image = val_ds[0]["image"].convert("RGB")
print(f"Predicting for label: {id2label[val_ds[0]['label']]}")

# Preprocess it
inputs = processor(images=one_image, return_tensors="pt").to(model.device)

# Get logits
with torch.no_grad():
    logits = model(**inputs).logits

# Get predicted class
predicted_class_idx = logits.argmax(-1).item()
print(f"Predicted class: {id2label[predicted_class_idx]}")


--- Running Inference Example ---
Predicting for label: Corn_(maize)___healthy
Predicted class: Corn_(maize)___healthy


### 15. Download the trained model to Google Drive

First, we'll zip the directory containing the saved model checkpoints. The `output_dir` was set to `./vit-plant-disease-results` in the `TrainingArguments`.

In [ ]:
import shutil
import os
from google.colab import drive

# Define the directory to be zipped
output_dir = "/content/vit-plant-disease-results/checkpoint-320"

# Define the name of the zip file
zip_filename = "vit-plant-disease-model"

# Create a zip archive of the output directory
print(f"Creating zip archive of {output_dir}...")
shutil.make_archive(zip_filename, 'zip', output_dir)
print(f"Archive '{zip_filename}.zip' created.")

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define the destination path in Google Drive
drive_destination_path = '/content/drive/MyDrive/vit-plant-disease-model.zip'

# Copy the zip file to Google Drive
print(f"Copying '{zip_filename}.zip' to Google Drive at '{drive_destination_path}'...")
shutil.copy(f'{zip_filename}.zip', drive_destination_path)
print("Model successfully copied to Google Drive!")

Creating zip archive of /content/vit-plant-disease-results/checkpoint-320...
Archive 'vit-plant-disease-model.zip' created.
Mounting Google Drive...
Mounted at /content/drive
Copying 'vit-plant-disease-model.zip' to Google Drive at '/content/drive/MyDrive/vit-plant-disease-model.zip'...
Model successfully copied to Google Drive!


In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
mshrestha_plant_disease_augmented_dataset_path = kagglehub.dataset_download('mshrestha/plant-disease-augmented-dataset')

print('Data source import complete.')


100%|██████████| 8.78G/8.78G [06:52<00:00, 22.9MB/s]

Extracting files...


Data source import complete.


In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
print(mshrestha_plant_disease_augmented_dataset_path)

/root/.cache/kagglehub/datasets/mshrestha/plant-disease-augmented-dataset/versions/1


In [ ]:
import os
os.listdir(mshrestha_plant_disease_augmented_dataset_path)

In [7]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Replace with your actual dataset root path
dataset_root = mshrestha_plant_disease_augmented_dataset_path

# Define your desired output structure
output_root = "/content/nepal_dataset"
train_dir = os.path.join(output_root, "train")
val_dir = os.path.join(output_root, "validation")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)


In [8]:
# Loop through each class folder
for class_name in os.listdir(dataset_root):
    class_path = os.path.join(dataset_root, class_name)

    if not os.path.isdir(class_path):
        continue  # skip any non-folder files

    # Make corresponding folders in train/ and validation/
    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)

    # Get all image files
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    # Split train/validation (80/20)
    train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

    # Copy train images
    for img in train_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(train_dir, class_name, img)
        shutil.copy(src, dst)

    # Copy validation images
    for img in val_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(val_dir, class_name, img)
        shutil.copy(src, dst)

print("✅ Dataset reorganized successfully!")
print(f"Train folder: {train_dir}")
print(f"Validation folder: {val_dir}")

✅ Dataset reorganized successfully!
Train folder: /content/nepal_dataset/train
Validation folder: /content/nepal_dataset/validation


In [4]:
import zipfile
import os

# Define the path to the zip file in Google Drive
zip_file_path = '/content/drive/MyDrive/vit-plant-disease-model.zip'

# Define the directory where the contents will be extracted
extraction_path = './extracted_model'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

# Extract the zip file
print(f"Extracting '{zip_file_path}' to '{extraction_path}'...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)
print("Extraction complete!")

Extracting '/content/drive/MyDrive/vit-plant-disease-model.zip' to './extracted_model'...
Extraction complete!


In [9]:
# --- KEY CHANGES START HERE ---
# 1. Define the path to your model from STAGE 1
# This should be the path to the best checkpoint saved by the previous Trainer.
# It will look something like "./vit-plant-disease-results/checkpoint-XXXX"
model_from_stage1_path = "/content/extracted_model" # <--- CHANGE THIS PATH

# 2. Define the path to your organized Nepali dataset
nepali_dataset_path = "/content/nepal_dataset" # <--- CHANGE THIS PATH


In [12]:
# 3. Load your custom Nepali dataset using "imagefolder"
nepali_dataset = load_dataset(
    "imagefolder",
    data_dir=nepali_dataset_path
)

# You now have `nepali_dataset['train']` and `nepali_dataset['validation']`
train_ds = nepali_dataset["train"].shuffle(seed=42)
val_ds = nepali_dataset["validation"].shuffle(seed=42)



Resolving data files:   0%|          | 0/119894 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29994 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [13]:
# 4. Get Labels from YOUR new dataset
labels = train_ds.features["label"].names
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}
num_labels = len(labels)
print(f"Loaded Nepali dataset with {num_labels} labels.")
print(f"Example labels: {labels[:5]}")



Loaded Nepali dataset with 34 labels.
Example labels: ['APPLE_HEALTHY', 'APPLE_ROT', 'APPLE_RUST', 'APPLE_SCAB', 'BANANA_HEALTHY']


In [14]:
print(labels)

['APPLE_HEALTHY', 'APPLE_ROT', 'APPLE_RUST', 'APPLE_SCAB', 'BANANA_HEALTHY', 'BANANA_PANAMA', 'BANANA_SIGATOKA', 'CORN_HEALTHY', 'CORN_LEAF_BLIGHT', 'CORN_LEAF_GRAY_SPOT', 'CORN_LEAF_RUST', 'PEPPER_BELL_BACTERIAL_SPOT', 'PEPPER_BELL_HEALTHY', 'POTATO_EARLY_BLIGHT', 'POTATO_HEALTHY', 'POTATO_LATE_BLIGHT', 'RICE_HEALTHY', 'RICE_LEAF_BLAST', 'RICE_LEAF_BLIGHT', 'RICE_LEAF_BROWN_SPOT', 'STRAWBERRY_HEALTHY', 'STRAWBERRY_LEAF_SCORCH', 'TEA_ALGAL_SPOT', 'TEA_BROWN_BLIGHT', 'TEA_HEALTHY', 'TEA_RED_LEAF_SPOT', 'TOMATO_BACTERIAL_SPOT', 'TOMATO_EARLY_BLIGHT', 'TOMATO_HEALTHY', 'TOMATO_LATE_BLIGHT', 'TOMATO_LEAF_MOLD', 'TOMATO_MOSAIC_VIRUS', 'TOMATO_SEPTORIA_LEAF_SPOT', 'TOMATO_TARGET_SPOT']


In [16]:
# --- PROCESSING REMAINS THE SAME ---

# 5. Load the Image Processor from the Stage 1 model
# This ensures we use the exact same preprocessing (resizing, normalization)
processor = AutoImageProcessor.from_pretrained(model_from_stage1_path, use_fast=True)



In [17]:
# 6. Define Image Transformations (can be the same as before)
normalize = Normalize(mean=processor.image_mean, std=processor.image_std)
_transforms = Compose(
    [
        RandomResizedCrop(processor.size["height"]),
        RandomHorizontalFlip(),
        ToTensor(),
        normalize,
    ]
)

def preprocess_func(examples):
    examples["pixel_values"] = [_transforms(image.convert("RGB")) for image in examples["image"]]
    return examples

train_ds.set_transform(preprocess_func)
val_ds.set_transform(preprocess_func)

In [18]:
# --- MODEL LOADING AND TRAINING ARGS ARE MODIFIED ---

# 7. Load the Fine-Tuned Model from Stage 1
# We are NOT starting from the original Google model.
model = AutoModelForImageClassification.from_pretrained(
    model_from_stage1_path,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True, # Important if your class count is different
)



Some weights of ViTForImageClassification were not initialized from the model checkpoint at /content/extracted_model and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([38]) in the checkpoint and torch.Size([34]) in the model instantiated
- classifier.weight: found shape torch.Size([38, 768]) in the checkpoint and torch.Size([34, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
# 8. Define NEW Training Configuration for Stage 2
# It's good practice to use a lower learning rate for the second fine-tuning
training_args = TrainingArguments(
    output_dir="./vit-nepali-finetuned-results", # New output directory
    per_device_train_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5, # Fine-tuning might need more epochs on the smaller dataset
    learning_rate=1e-5, # LOWER learning rate for fine-tuning an already trained model
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,
    report_to="none",

)


In [21]:
# --- TRAINER SETUP REMAINS THE SAME ---

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.tensor([x["label"] for x in batch]),
    }





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

/tmp/ipython-input-3264422827.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 11. Start Stage 2 Training!
print("Starting Stage 2 fine-tuning on Nepali dataset...")
trainer.train()
print("Training finished.")



Starting Stage 2 fine-tuning on Nepali dataset...


Epoch,Training Loss,Validation Loss


In [ ]:
# 12. Evaluate your final, specialized model
print("Evaluating the final Nepal-specific model...")
eval_results = trainer.evaluate()
print(f"Final evaluation results on Nepali validation set: {eval_results}")